# EHUNAM on Kaggle - Phase 0 discovery

EHUNAM (figshare 28541225, Scientific Data 2025): 23 days, 8 environments, up to 21 people and 9 machines;
tasks People Counting (PC), Human Activity Recognition (HAR), Machine Recognition (MR), Machine Activity
Recognition (MAR). 2,401 `.mat` files + `Summary.xlsx` / `Summary.mat` describing each file.

The paper with the exact file layout is not freely readable, so this notebook **discovers** it:
1. reads `Summary.xlsx` (columns, counts, people-count column);
2. splits the file names into their codes and matches them with the summary;
3. opens one file of each code and prints every variable, shape and dtype;
4. if Kaggle kept the upload as `EHUNAM.zip`, lists it and extracts a few members (Deflate64).

**Save the notebook after running** and share the output, so the Phase 1 converter can be written for EHUNAM.

In [ ]:
# 1. Setup and discovery of the uploaded files
import glob
import os
import re
import zipfile
from collections import Counter

import h5py
import numpy as np
import pandas as pd
import scipy.io as sio

# Locate dataset folders under /kaggle/input (works for any dataset slug / nesting)
SKIP_DIRS = {"amp", "mat", "video"}            # huge leaf folders: never walk into them


def find_dirs(test, root="/kaggle/input", max_depth=6):
    """Return folders for which test(folder, file_names) is True (shallow walk)."""
    hits = []
    base = root.rstrip("/").count("/")
    for folder, subdirs, files in os.walk(root):
        if test(folder, files):
            hits.append(folder)
        if folder.count("/") - base >= max_depth:
            subdirs[:] = []
        subdirs[:] = [s for s in subdirs if s not in SKIP_DIRS and not s.startswith("sub_")]
    return sorted(hits)


print("attached datasets:", sorted(os.listdir("/kaggle/input")))

summary_dirs = find_dirs(lambda folder, files: "Summary.xlsx" in files or "Summary.mat" in files)
zip_dirs = find_dirs(lambda folder, files: "EHUNAM.zip" in files)
mat_dirs = find_dirs(lambda folder, files: any(f.endswith(".mat") and not f.startswith("Summary") for f in files))
print("Summary in :", summary_dirs)
print("zip in     :", zip_dirs)
print(".mat in    :", mat_dirs[:10], "..." if len(mat_dirs) > 10 else "")

WORK = "/kaggle/working/ehunam_explore"
os.makedirs(WORK, exist_ok=True)

In [ ]:
# 2. Summary.xlsx: columns, value counts, people-count column
assert summary_dirs, "Summary.xlsx not found - is the EHUNAM upload finished and attached?"
xlsx = os.path.join(summary_dirs[0], "Summary.xlsx")
sheets = pd.read_excel(xlsx, sheet_name=None)
for name, df in sheets.items():
    print("=" * 100)
    print("sheet", name, df.shape)
    print(list(df.columns))
    display(df.head(10))
    for c in df.columns:
        n = df[c].nunique()
        if 1 < n <= 40:
            print(f"  {c!r}: {df[c].value_counts().sort_index().to_dict()}")
        else:
            print(f"  {c!r}: {n} unique, e.g. {df[c].dropna().astype(str).head(3).tolist()}")

summary = next(iter(sheets.values()))
summary.to_csv(os.path.join(WORK, "summary_sheet1.csv"), index=False)

smat = os.path.join(summary_dirs[0], "Summary.mat")
if os.path.exists(smat):
    s = sio.loadmat(smat, squeeze_me=True, struct_as_record=False)
    print("Summary.mat variables:", {k: type(v).__name__ for k, v in s.items() if not k.startswith("__")})

In [ ]:
# 3. File names -> codes (MCx_<id>_<n>_<task code>_...), matched against the summary
if mat_dirs:
    names = sorted(os.path.basename(p) for d in mat_dirs for p in glob.glob(os.path.join(d, "*.mat"))
                   if not os.path.basename(p).startswith("Summary"))
    paths = {os.path.basename(p): p for d in mat_dirs for p in glob.glob(os.path.join(d, "*.mat"))}
    SOURCE = "files"
elif zip_dirs:
    zf = zipfile.ZipFile(os.path.join(zip_dirs[0], "EHUNAM.zip"))
    infos = [i for i in zf.infolist() if i.filename.endswith(".mat")]
    names = sorted(os.path.basename(i.filename) for i in infos)
    print("zip entries:", len(infos), "| compression methods:", Counter(i.compress_type for i in infos))
    SOURCE = "zip"
else:
    raise SystemExit("no EHUNAM .mat files or EHUNAM.zip attached yet")

print(len(names), "mat files; first:", names[:5])
tokens = pd.DataFrame([n[:-4].split("_") for n in names])
tokens.insert(0, "name", names)
for c in tokens.columns[1:8]:
    vc = tokens[c].value_counts()
    print(f"token {c}: {len(vc)} unique  {vc.head(15).to_dict()}")

TASK_CODES = ["PCMAR", "PC", "HAR", "MAR", "MR", "E"]
tokens["code"] = [next((t for t in TASK_CODES if f"_{t}_" in f"_{n[:-4]}_"), "?") for n in names]
print("task codes:", tokens["code"].value_counts().to_dict())

# which summary column holds the file names?
name_stems = set(n[:-4] for n in names)
for c in summary.columns:
    hit = summary[c].astype(str).str.replace(".mat", "", regex=False).isin(name_stems).mean()
    if hit > 0.1:
        print(f"summary column {c!r} matches {hit:.0%} of rows to file names")
tokens.to_csv(os.path.join(WORK, "file_tokens.csv"), index=False)

In [ ]:
# 4. Open one file per task code: every variable, shape, dtype, small value sample
if SOURCE == "zip":
    !pip -q install zipfile-deflate64
    import zipfile_deflate64 as zipfile64                   # handles Deflate64 (method 9)
    zf64 = zipfile64.ZipFile(os.path.join(zip_dirs[0], "EHUNAM.zip"))
    by_name = {os.path.basename(i.filename): i.filename for i in zf64.infolist()}


def local_path(name):
    if SOURCE == "files":
        return paths[name]
    out = os.path.join(WORK, name)
    if not os.path.exists(out):
        with zf64.open(by_name[name]) as src, open(out, "wb") as dst:
            dst.write(src.read())
    return out


def describe(name, value, depth=0):
    pad = "    " * depth
    if isinstance(value, np.ndarray) and value.dtype.names:
        print(f"{pad}{name}: struct array {value.shape} fields={value.dtype.names}")
        for field in value.dtype.names[:20]:
            describe(field, value[field].flat[0] if value.size else None, depth + 1)
    elif isinstance(value, np.ndarray):
        sample = value.ravel()[:6]
        print(f"{pad}{name}: {value.dtype} {value.shape}  e.g. {sample}")
    elif isinstance(value, h5py.Dataset):
        print(f"{pad}{name}: h5 {value.dtype} {value.shape}")
    elif isinstance(value, h5py.Group):
        print(f"{pad}{name}: h5 group {list(value.keys())[:20]}")
        for k in list(value.keys())[:20]:
            describe(k, value[k], depth + 1)
    else:
        print(f"{pad}{name}: {type(value).__name__} {str(value)[:80]}")


for code, g in tokens.groupby("code"):
    name = g["name"].iloc[0]
    path = local_path(name)
    print("=" * 100)
    print(code, name, f"{os.path.getsize(path) / 1e6:.1f} MB")
    try:
        m = sio.loadmat(path)
        for k, v in m.items():
            if not k.startswith("__"):
                describe(k, v, 1)
    except NotImplementedError:                              # MATLAB v7.3 = HDF5
        with h5py.File(path, "r") as f:
            for k in f.keys():
                describe(k, f[k], 1)